In [ ]:
import sempy.fabric as fabric
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import os
import re
from glob import glob
from datetime import datetime
from pyspark.sql.functions import lit

In [ ]:
import sempy.fabric as fabric
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import networkx as nx
import numpy as np
from collections import defaultdict

In [ ]:

%run 2. Parameters

In [ ]:
DATASET   = semantic_model_name
WORKSPACE = workspaceId
OUTPUT_PNG = "/lakehouse/default/Files/er_diagram.png"
DPI        = 300

In [ ]:
STYLE = {
    "bg_color":         "#FFFFFF",
    "header_bg":        "#2B3A4E",
    "header_font":      "white",
    "header_fontsize":  10,
    "col_bg":           "#F5F7FA",
    "col_font":         "#2B3A4E",
    "col_fontsize":     8,
    "key_color":        "#D4A017",      # Farbe für FK/PK-Spalten
    "box_border":       "#2B3A4E",
    "box_radius":       0.02,
    "line_color":       "#666666",
    "line_inactive":    "#CC4444",
    "line_bidir":       "#D4A017",
    "mult_fontsize":    8,
    "mult_color":       "#2B3A4E",
    "box_width":        2.2,
    "row_height":       0.30,
    "header_height":    0.40,
    "box_padding":      0.10,
}

In [ ]:
df_rel = fabric.list_relationships(datasetId, workspace=WORKSPACE)
df_col = fabric.list_columns(datasetId, workspace=WORKSPACE)

auto_tables = df_rel["From Table"].str.startswith(("LocalDateTable_", "DateTableTemplate_")) | \
              df_rel["To Table"].str.startswith(("LocalDateTable_", "DateTableTemplate_"))
df_rel = df_rel[~auto_tables].reset_index(drop=True)

In [ ]:
table_columns = defaultdict(list)
for _, row in df_col.iterrows():
    tbl = row.get("Table Name") or row.get("tableName") or row.get("table_name", "")
    col = row.get("Column Name") or row.get("columnName") or row.get("column_name", "")
    dtype = row.get("Data Type") or row.get("dataType") or row.get("data_type", "")
    if tbl and col:
        table_columns[tbl].append({"name": col, "type": str(dtype)})

In [ ]:
rel_tables = set(df_rel["From Table"].unique()) | set(df_rel["To Table"].unique())
table_columns = {t: cols for t, cols in table_columns.items() if t in rel_tables}

In [ ]:
# Falls Tabellen in Relationen sind, aber keine Spalten gefunden wurden
for t in rel_tables:
    if t not in table_columns:
        table_columns[t] = []

In [ ]:
# FK-Spalten markieren
fk_columns = set()
for _, row in df_rel.iterrows():
    fk_columns.add((row["From Table"], row["From Column"]))
    fk_columns.add((row["To Table"],   row["To Column"]))

In [ ]:
def calc_box_width(table_name):
    """Berechnet die Box-Breite basierend auf dem längsten Spalteninhalt."""
    columns = table_columns.get(table_name, [])
    
    # Längster Spaltenname + Datentyp
    max_col_len = len(table_name)  # Mindestens so breit wie der Header
    for col in columns:
        col_text = f"[PK] {col['name']}   {col['type']}" if (table_name, col['name']) in fk_columns else f"    {col['name']}   {col['type']}"
        max_col_len = max(max_col_len, len(col_text))
    
    # Umrechnung: ca. 0.12 Einheiten pro Zeichen, Minimum 2.2
    return max(3.0, max_col_len * 0.15 + 0.5)

In [ ]:
# Multiplicity-Mapping
MULT_MAP = {
    "onetoone":     "1 : 1",   "onetomany":    "1 : n",
    "manytoone":    "n : 1",   "manytomany":   "m : n",
    "one-to-one":   "1 : 1",   "one-to-many":  "1 : n",
    "many-to-one":  "n : 1",   "many-to-many": "m : n",
    "*:1":          "n : 1",   "1:*":          "1 : n",
    "1:1":          "1 : 1",   "*:*":          "m : n",
    "m:1":          "m : 1",   "1:m":          "1 : m"
}

def friendly_mult(raw) -> str:
    if raw is None:
        return ""
    return MULT_MAP.get(str(raw).strip().lower(), str(raw))

In [ ]:
G = nx.Graph()
for t in table_columns:
    G.add_node(t)
for _, row in df_rel.iterrows():
    G.add_edge(row["From Table"], row["To Table"])

pos = nx.spring_layout(G, k=10.0 / max(np.sqrt(len(table_columns)), 1),
                        iterations=200, seed=42, scale=10)

In [ ]:
def box_height(table_name):
    n_cols = len(table_columns.get(table_name, []))
    n_cols = max(n_cols, 1)  # Mindestens 1 Zeile
    header_gap = 0.15
    return STYLE["header_height"] + header_gap + n_cols * STYLE["row_height"] + STYLE["box_padding"]

In [ ]:
fig_w = max(16, len(table_columns) * 3.5)
fig_h = max(10, len(table_columns) * 2.0)
fig, ax = plt.subplots(figsize=(fig_w, fig_h), facecolor=STYLE["bg_color"])
ax.set_facecolor(STYLE["bg_color"])
ax.set_aspect("equal")
ax.axis("off")



ax.set_title(f"{DATASET}", fontsize=16, fontweight="bold",
             color=STYLE["header_bg"], pad=20, fontfamily="sans-serif")

box_positions = {}   # {table: (x_center, y_center, width, height)}
col_positions = {}   # {(table, col): (x_center, y_row)}

for table_name in table_columns:
    cx, cy = pos[table_name]
    w = calc_box_width(table_name)
    h = box_height(table_name)
    x0 = cx - w / 2
    y0 = cy - h / 2

    box_positions[table_name] = (cx, cy, w, h)

    bg_box = FancyBboxPatch(
        (x0, y0), w, h,
        boxstyle=f"round,pad={STYLE['box_radius']}",
        facecolor=STYLE["col_bg"],
        edgecolor=STYLE["box_border"],
        linewidth=1.5,
        zorder=2,
    )
    ax.add_patch(bg_box)

    header_h = STYLE["header_height"]
    header_y = y0 + h - header_h
    header_box = FancyBboxPatch(
        (x0, header_y), w, header_h,
        boxstyle=f"round,pad={STYLE['box_radius']}",
        facecolor=STYLE["header_bg"],
        edgecolor=STYLE["box_border"],
        linewidth=1.5,
        zorder=3,
    )
    ax.add_patch(header_box)

    ax.text(
        cx, header_y + header_h / 2,
        table_name,
        ha="center", va="center",
        fontsize=STYLE["header_fontsize"],
        fontweight="bold",
        color=STYLE["header_font"],
        fontfamily="sans-serif",
        zorder=4,
    )

    columns = table_columns.get(table_name, [])
    if not columns:
        columns = [{"name": "(keine Spalten)", "type": ""}]

    for i, col_info in enumerate(columns):
        col_name = col_info["name"]
        col_type = col_info["type"]
        header_gap = 0.15  # Abstand zwischen Header und erster Spalte
        row_y = header_y - header_gap - (i + 1) * STYLE["row_height"] + STYLE["row_height"] / 2

        is_key = (table_name, col_name) in fk_columns
        prefix = "► " if is_key else "    "
        font_color = STYLE["key_color"] if is_key else STYLE["col_font"]
        font_weight = "bold" if is_key else "normal"

        ax.text(
            x0 + 0.08, row_y,
            f"{prefix}{col_name}",
            ha="left", va="center",
            fontsize=STYLE["col_fontsize"],
            color=font_color,
            fontweight=font_weight,
            fontfamily="sans-serif",
            zorder=4,
        )

        if col_type:
            short_type = col_type.replace("System.", "").replace("Int64", "Int")
            ax.text(
                x0 + w - 0.08, row_y,
                short_type,
                ha="right", va="center",
                fontsize=STYLE["col_fontsize"] - 1,
                color="#999999",
                fontstyle="italic",
                fontfamily="sans-serif",
                zorder=4,
            )

        col_positions[(table_name, col_name)] = (cx, row_y)

ax.plot([x0 + 0.03, x0 + w - 0.03], [header_y, header_y], color=STYLE["box_border"], linewidth=1, zorder=3);

In [ ]:
def edge_point(table_name, target_x):
    """Berechnet den Verbindungspunkt am Rand der Box (links oder rechts)."""
    cx, cy, w, h = box_positions[table_name]
    if target_x < cx:
        return cx - w / 2, cy  # linke Seite
    else:
        return cx + w / 2, cy  # rechte Seite

In [ ]:
for _, row in df_rel.iterrows():
    from_table = row["From Table"]
    to_table   = row["To Table"]
    from_col   = row["From Column"]
    to_col     = row["To Column"]

    mult_raw   = row.get("Multiplicity", "")
    mult_label = friendly_mult(mult_raw)

    cross_raw  = str(row.get("Cross Filtering Behavior", "")).strip().lower()
    is_bidir   = cross_raw in ("bothdirections", "both", "bidirectional")
    is_active  = row.get("isActive", True)

    from_cx, from_cy, from_w, _ = box_positions[from_table]
    to_cx,   to_cy,   to_w,   _ = box_positions[to_table]

    if from_cx < to_cx:
        x1 = from_cx + from_w / 2
        x2 = to_cx   - to_w  / 2
    else:
        x1 = from_cx - from_w / 2
        x2 = to_cx   + to_w  / 2

    y1 = col_positions.get((from_table, from_col), (0, from_cy))[1]
    y2 = col_positions.get((to_table,   to_col),   (0, to_cy)  )[1]

    if not is_active:
        lcolor = STYLE["line_inactive"]
        lstyle = "--"
    elif is_bidir:
        lcolor = STYLE["line_bidir"]
        lstyle = "-"
    else:
        lcolor = STYLE["line_color"]
        lstyle = "-"

    # ── build xs / ys FIRST, then derive midpoint ──────────────────────
    if abs(from_cx - to_cx) < 0.3:
        from_w_half = box_positions[from_table][2] / 2
        to_w_half   = box_positions[to_table][2]   / 2
        x1 = from_cx + from_w_half
        x2 = to_cx   + to_w_half
        detour_x = max(x1, x2) + 0.8
        xs = [x1, detour_x, detour_x, x2]
        ys = [y1, y1,       y2,       y2]
        arrow_dx = -0.12
    else:
        mid_x_line = (x1 + x2) / 2
        xs = [x1, mid_x_line, mid_x_line, x2]
        ys = [y1, y1,         y2,         y2]
        arrow_dx = 0.12 if x2 > x1 else -0.12

    # ── now xs / ys exist ───────────────────────────────────────────────
    mid_x  = (xs[1] + xs[2]) / 2
    mid_y  = (ys[1] + ys[2]) / 2
    offset = 0.15 * np.sign(y1 - y2 + 0.001)   # kept in case you use it later

    ax.plot(xs, ys,
            color=lcolor, linestyle=lstyle, linewidth=1.5,
            zorder=1, solid_capstyle="round")

    ax.annotate("",
                xy=(x2, y2),
                xytext=(x2 - arrow_dx, y2),
                arrowprops=dict(arrowstyle="-|>", color=lcolor, linewidth=1.5),
                zorder=1)

    label_parts = []
    if mult_label:          label_parts.append(mult_label)
    if is_bidir:            label_parts.append("⇄")
    if not is_active:       label_parts.append("(inaktiv)")
    label_text = " ".join(label_parts)

    if label_text:
        ax.text(
            mid_x, mid_y + 0.25, label_text,
            ha="center", va="center",
            fontsize=STYLE["mult_fontsize"], fontweight="bold",
            color=STYLE["mult_color"], fontfamily="sans-serif",
            bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                      edgecolor=lcolor, linewidth=0.8, alpha=0.95),
            zorder=5,
        )

In [ ]:
legend_items = [
    mpatches.Patch(facecolor=STYLE["line_color"],    label="Active Relationship"),
    mpatches.Patch(facecolor=STYLE["line_bidir"],    label="Bidirectional"),
    mpatches.Patch(facecolor=STYLE["line_inactive"], label="Inactive"),
    mpatches.Patch(facecolor=STYLE["key_color"],     label="► Relationship Column"),
]
ax.legend(handles=legend_items, loc="lower right", fontsize=8,
          facecolor="white", edgecolor="#cccccc", framealpha=0.9)

In [ ]:
plt.tight_layout()

fig.savefig(OUTPUT_PNG, dpi=DPI, bbox_inches="tight", facecolor=STYLE["bg_color"])
print(f"✅  PNG gespeichert: {OUTPUT_PNG}")

In [ ]:
#remove this 
all_x = [p[0] for p in pos.values()]
all_y = [p[1] for p in pos.values()]
print(f"X range: {min(all_x):.2f} to {max(all_x):.2f}")
print(f"Y range: {min(all_y):.2f} to {max(all_y):.2f}")
print(f"xlim will be: {min(all_x)-3:.2f} to {max(all_x)+3:.2f}")
print(f"ylim will be: {min(all_y)-3:.2f} to {max(all_y)+3:.2f}")

In [ ]:
fig.tight_layout()

# -----------------------------
# SAVE WITH VERSIONING (starting from v1)
# -----------------------------
folder_path = "/lakehouse/default/Files/diagrams"
os.makedirs(folder_path, exist_ok=True)

clean_name = re.sub(r'[^\w\-]', '_', semantic_model_name)
base_filename = f"semantic_model_{workspaceId}_{datasetId}_{clean_name}"

existing_files = glob(f"{folder_path}/{base_filename}_v*.jp*") + \
                 glob(f"{folder_path}/{base_filename}_v*.png")

versions = []
for filepath in existing_files:
    fname = os.path.basename(filepath)
    match = re.search(r'_v(\d+)\.(png|jpg|jpeg)$', fname)
    if match:
        versions.append(int(match.group(1)))

if versions:
    next_version = max(versions) + 1
else:
    next_version = 1

filename = f"{base_filename}_v{next_version}.jpg"
full_path = f"{folder_path}/{filename}"

# -----------------------------
# ✅ FORCE WHITE BACKGROUND ON FIGURE AND ALL AXES
# -----------------------------
from PIL import Image
import os

fig.patch.set_facecolor('white')        # ← Figure outer background
for ax in fig.get_axes():
    ax.set_facecolor('white')           # ← Every axes/subplot background

# -----------------------------
# SAVE AS JPEG — Forces RGB, no alpha possible
# -----------------------------
fig.savefig(full_path,
            dpi=150,              # ← Lower DPI, smaller file
            bbox_inches="tight",
            pad_inches=0,         # ← Zero padding
            facecolor='white',
            edgecolor='none',
            format='jpeg')

print(f"Saved to: {full_path}")

# -----------------------------
# DIAGNOSTIC CHECKS
# -----------------------------
file_size = os.path.getsize(full_path)
img_check = Image.open(full_path)

print(f"Saved file size: {file_size:,} bytes")
print(f"Image size: {img_check.size}")
print(f"Image mode: {img_check.mode}")   # Must show RGB ✅

if img_check.mode != 'RGB':
    raise ValueError(f"❌ Unexpected mode: {img_check.mode}")

print("✅ RGB confirmed - white background applied")

# -----------------------------
# READ IMAGE AS BINARY
# -----------------------------
with open(full_path, 'rb') as f:
    image_binary = f.read()

print(f"Binary size: {len(image_binary):,} bytes")

# -----------------------------
# GET ONELAKE URL
# -----------------------------
try:
    lakehouse_id = spark.conf.get("trident.lakehouse.id")
    workspace_id = spark.conf.get("trident.workspace.id")
    
    relative_path = f"Files/diagrams/{filename}"
    onelake_url = f"https://onelake.dfs.fabric.microsoft.com/{workspace_id}/{lakehouse_id}/{relative_path}"
    
    print(f"\n{'='*80}")
    print(f"OneLake URL: {onelake_url}")
    print(f"{'='*80}\n")
    print(f"Workspace ID: {workspace_id}")
    print(f"Lakehouse ID: {lakehouse_id}")
    
except Exception as e:
    print(f"Could not retrieve lakehouse/workspace IDs: {e}")
    onelake_url = None

image_count = len(glob(f"{folder_path}/*.jpg"))
print(f"Total images in folder: {image_count}")

# -----------------------------
# CREATE/UPDATE OUTPUT TABLE WITH BINARY AND BASE64 CHUNKS
# -----------------------------
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BinaryType
from pyspark.sql.functions import lit
from datetime import datetime
import base64 as base64_lib

image_base64_str = base64_lib.b64encode(image_binary).decode('utf-8')

# -----------------------------
# SPLIT BASE64 INTO EXACTLY 11 CHUNKS
# -----------------------------
NUM_CHUNKS = 11
base64_chunks = []
total_length = len(image_base64_str)

chunk_size = (total_length + NUM_CHUNKS - 1) // NUM_CHUNKS

for i in range(NUM_CHUNKS):
    start = i * chunk_size
    end = min(start + chunk_size, total_length)
    chunk = image_base64_str[start:end]
    base64_chunks.append(chunk)

num_chunks = NUM_CHUNKS

print(f"\n📊 Base64 Split Info:")
print(f"  - Total Base64 length: {total_length:,} characters")
print(f"  - Number of chunks: {num_chunks}")
print(f"  - Chunk size: {chunk_size:,} characters")
for i, chunk in enumerate(base64_chunks, 1):
    print(f"  - Chunk {i} length: {len(chunk):,} characters")



# Create schema
schema_fields = [
    StructField("WorkspaceId", StringType(), True),
    StructField("DatasetId", StringType(), True),
    StructField("SemanticModelName", StringType(), True),
    StructField("NoOfTablesWithRelationships", IntegerType(), True),   # ← ADD HERE
    StructField("TotalTables", IntegerType(), True),
    StructField("Version", IntegerType(), True),
    StructField("FileName", StringType(), True),
    StructField("FolderPath", StringType(), True),
    StructField("BaseFileName", StringType(), True),
    StructField("FullPath", StringType(), True),
    StructField("OneLakeURL", StringType(), True),
    ##StructField("ImageBinary", BinaryType(), True),
    StructField("ImageBase64Part1", StringType(), True),
    StructField("ImageBase64Part2", StringType(), True),
    StructField("ImageBase64Part3", StringType(), True),
    StructField("ImageBase64Part4", StringType(), True),
    StructField("ImageBase64Part5", StringType(), True),
    StructField("ImageBase64Part6", StringType(), True),
    StructField("ImageBase64Part7", StringType(), True),
    StructField("ImageBase64Part8", StringType(), True),
    StructField("ImageBase64Part9", StringType(), True),
    StructField("ImageBase64Part10", StringType(), True),
    StructField("ImageBase64Part11", StringType(), True),
    StructField("TotalChunks", IntegerType(), True),
    StructField("MIMEType", StringType(), True),
    StructField("CreatedDate", StringType(), True)
]

schema = StructType(schema_fields)

data_row = [
    workspaceId,
    datasetId,
    semantic_model_name,
    len(table_columns),
    len(df_col["Table Name"].unique()),
    next_version,
    filename,
    folder_path,
    base_filename,
    full_path,
    onelake_url,
    ##bytearray(image_binary),
    base64_chunks[0],
    base64_chunks[1],
    base64_chunks[2],
    base64_chunks[3],
    base64_chunks[4],
    base64_chunks[5],
    base64_chunks[6],
    base64_chunks[7],
    base64_chunks[8],
    base64_chunks[9],
    base64_chunks[10],
    num_chunks,
    "image/jpeg",
    datetime.now().strftime("%Y-%m-%d %H:%M:%S")
]

df_output = spark.createDataFrame([tuple(data_row)], schema)

df_output = df_output.withColumn("DATABASE_TIMESTAMP",
                                  lit(f"{semantic_model_name} | {datetime.now():%d.%m.%Y %H:%M:%S}"))
df_output = df_output.withColumn('DATABASE_VERSION',
                                  lit(f"{semantic_model_name} | V{next_version}"))

table_name = "diagrams"

if spark.catalog.tableExists(table_name):
    write_mode = "append"
else:
    write_mode = "overwrite"

df_output.write.format("delta") \
    .mode(write_mode) \
    .option("mergeSchema", "true") \
    .saveAsTable(table_name)

print(f"\n✓ Record added to table: {table_name}")
print(f"  - Binary image size: {len(image_binary):,} bytes")
print(f"  - Base64 total length: {len(image_base64_str):,} characters")
print(f"  - Split into {num_chunks} chunks")
print(f"  - Version: V{next_version}")
print(f"  - Write mode: {write_mode}")

display(df_output)
display(fig)